# Phase 5: Automated scoring pipeline — raw data collection

**Goal:** run GR00T N1.7 and Cosmos-Reason2-2B across a batch of episodes and log their
raw outputs (predicted action chunks, actual recorded actions, reasoning traces) to
Google Drive as JSON — one file per episode. This notebook does **not** implement the
actual scoring rules (match/minor-deviation/failure, the reasoning-execution 2x2, etc.)
— those live in `scripts/score_trajectories.py`, which runs locally and reads the JSON
this notebook produces. Splitting it this way keeps GPU-only work on Colab and moves
everything that doesn't need a GPU (scoring math, an LLM-as-judge call for reasoning
matching) off of scarce Colab time.

See `docs/scoring_rubric.md` for the full rule definitions this pipeline's output feeds
into.

**Design note — three observation points per episode, not one.** The Phase 3 smoke test
queried GR00T once, at frame 0. To get a real per-phase reasoning-vs-execution
comparison (the project's core "intent lost in handoff" 2x2 in the rubric), GR00T is
queried **three times per episode** here: once at the start of each derived sub-goal
(reach → transport → retreat, using the grasp/release frame indices from the hand-closure
segmentation below). Cosmos-Reason2 still runs only **once** per episode, since it
produces one full step-by-step plan up front, not a new plan per phase. This roughly
triples GR00T's per-episode compute cost but is what actually lets the scoring script
tell whether a good plan and a good hand-off held up through transport and retreat, not
just at the very first step.

**If any cell errors: stop, copy the full error output, and paste it back.** Same rule
as notebook 1 — don't try to patch it yourself.

## 0. Drive mount + checkpoint scaffolding

Separate checkpoint file and raw-log directory from notebook 1 (`phase5_status.json`,
`raw_logs/`), so this can be re-run independently without touching Phase 2/3's status.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time

PROJECT_DIR = '/content/drive/MyDrive/humanoid-vla-eval'
CKPT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
DATA_DIR = os.path.join(PROJECT_DIR, 'data', 'g1_teleop_subset')
RAW_LOG_DIR = os.path.join(PROJECT_DIR, 'raw_logs')
STATUS_PATH = os.path.join(CKPT_DIR, 'phase5_status.json')

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RAW_LOG_DIR, exist_ok=True)

def load_status():
    if os.path.exists(STATUS_PATH):
        with open(STATUS_PATH) as f:
            return json.load(f)
    return {}

def save_status(key, value):
    status = load_status()
    status[key] = value
    status['_last_updated'] = time.strftime('%Y-%m-%d %H:%M:%S')
    with open(STATUS_PATH, 'w') as f:
        json.dump(status, f, indent=2)
    return status

def is_done(key):
    return load_status().get(key) == 'done'

print('Current status:', load_status())

## 1. Install dependencies

Identical to notebook 1's install cell — copied verbatim so this notebook can run in a
fresh Colab session on its own, without depending on notebook 1 having run first in the
same kernel.

In [ ]:
import os, subprocess, platform, importlib, sys

def run(cmd):
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed (exit {result.returncode}): {cmd}')

GR00T_REPO_PATH = '/content/Isaac-GR00T'
RESTART_MARKER = '/content/.numpy_clean_reinstall_done'

def clean_reinstall_numpy():
    result = subprocess.run(
        [sys.executable, '-c', 'import numpy, os; print(os.path.dirname(numpy.__file__))'],
        capture_output=True, text=True,
    )
    if result.returncode == 0 and result.stdout.strip():
        numpy_dir = result.stdout.strip()
        print(f'Found existing numpy install at: {numpy_dir} — removing it entirely.')
        run(f'rm -rf "{numpy_dir}"')
    else:
        print('No existing numpy import found (or already removed) — nothing to clean.')
    run(f'"{sys.executable}" -m pip uninstall -y numpy')
    run(f'uv pip install --python "{sys.executable}" --reinstall --no-cache numpy==1.26.4')
    with open(RESTART_MARKER, 'w') as f:
        f.write('numpy was reinstalled cleanly on disk; the CURRENT process may still have a '
                'stale copy loaded and needs a restart to pick it up.')

def gr00t_deep_import():
    if os.path.isdir(GR00T_REPO_PATH) and GR00T_REPO_PATH not in sys.path:
        sys.path.insert(0, GR00T_REPO_PATH)
    importlib.invalidate_caches()
    try:
        import gr00t  # noqa: F401
        import gr00t.policy  # noqa: F401
        import gr00t.data.embodiment_tags  # noqa: F401
        import pandas  # noqa: F401
        import pyarrow  # noqa: F401
        import cv2  # noqa: F401
        import pytorch_kinematics  # noqa: F401
        return True, False
    except ImportError:
        return False, False
    except ValueError as e:
        print(f'numpy ABI mismatch in this process: {e}')
        return False, True

def print_diagnostics():
    print('\n--- DIAGNOSTICS ---')
    print('Kernel sys.executable:', sys.executable)
    print()
    subprocess.run(f'"{sys.executable}" -m pip show gr00t numpy pyarrow pytorch-kinematics', shell=True)
    print()
    print('Fresh-subprocess import test (same interpreter as kernel):')
    result = subprocess.run([sys.executable, '-c', 'import gr00t.policy, pandas, pyarrow, cv2, pytorch_kinematics; import numpy; print("OK:", gr00t.__file__, numpy.__version__)'], capture_output=True, text=True)
    print('returncode:', result.returncode)
    print('stdout:', result.stdout)
    print('stderr:', result.stderr)

if os.path.exists(RESTART_MARKER):
    os.remove(RESTART_MARKER)

ok, needs_restart = gr00t_deep_import()

if ok:
    print('gr00t already importable — skipping install.')
    save_status('deps_installed', 'done')
elif needs_restart:
    print('numpy is already loaded in this running kernel in a way that cannot be fixed '
          'without a restart. Cleaning the on-disk install now so a restart picks up a '
          'consistent state immediately...')
    clean_reinstall_numpy()
    raise RuntimeError(
        'ACTION REQUIRED: Runtime > Restart session (NOT "Restart and run all" — just '
        'restart), then re-run this cell.'
    )
else:
    if is_done('deps_installed'):
        print("Checkpoint says deps_installed=done but the deep import fails — reinstalling for real.")

    py_ver = platform.python_version()
    print(f'Python version: {py_ver}')
    if not py_ver.startswith('3.12'):
        print('WARNING: Isaac-GR00T requires Python 3.12.x.')

    if not os.path.exists(GR00T_REPO_PATH):
        run(f'git clone https://github.com/NVIDIA/Isaac-GR00T.git {GR00T_REPO_PATH}')

    run('pip install -q -U uv')
    run(f'uv pip install --python "{sys.executable}" -e {GR00T_REPO_PATH}')
    run(f'uv pip install --python "{sys.executable}" accelerate pyarrow pytorch-kinematics')

    ok, needs_restart = gr00t_deep_import()
    if needs_restart:
        clean_reinstall_numpy()
        raise RuntimeError('ACTION REQUIRED: Runtime > Restart session, then re-run this cell.')
    elif not ok:
        print_diagnostics()
        raise RuntimeError(
            'Install commands reported success but the deep import still fails. Copy the '
            'full output of this cell back to Claude.'
        )

    save_status('deps_installed', 'done')
    print('Dependencies installed and verified importable.')

## 2. Hugging Face auth

In [ ]:
from huggingface_hub import login
login()

## 3. Load GR00T N1.7 (System 1 / action model)

In [ ]:
import torch

GR00T_CHECKPOINT = 'nvidia/GR00T-N1.7-3B'

from gr00t.policy import Gr00tPolicy
from gr00t.data.embodiment_tags import EmbodimentTag

gr00t_policy = Gr00tPolicy(
    model_path=GR00T_CHECKPOINT,
    embodiment_tag=EmbodimentTag.REAL_G1,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
gr00t_modality_configs = gr00t_policy.get_modality_config()
print('GR00T N1.7 loaded successfully with EmbodimentTag.REAL_G1.')
for modality, cfg in gr00t_modality_configs.items():
    print(f'{modality}: keys={cfg.modality_keys}, horizon={len(cfg.delta_indices)}')
save_status('gr00t_loaded', 'done')

## 4. Load Cosmos-Reason2-2B (System 2 reasoning proxy)

In [ ]:
COSMOS_REASON2_CHECKPOINT = 'nvidia/Cosmos-Reason2-2B'

import transformers

PIXELS_PER_TOKEN = 32 ** 2

cosmos_model = transformers.Qwen3VLForConditionalGeneration.from_pretrained(
    COSMOS_REASON2_CHECKPOINT,
    dtype=torch.float16,
    device_map='auto',
    attn_implementation='sdpa',
)
cosmos_processor = transformers.Qwen3VLProcessor.from_pretrained(COSMOS_REASON2_CHECKPOINT)

min_vision_tokens, max_vision_tokens = 256, 8192
cosmos_processor.image_processor.size = {
    'shortest_edge': min_vision_tokens * PIXELS_PER_TOKEN,
    'longest_edge': max_vision_tokens * PIXELS_PER_TOKEN,
}
cosmos_processor.video_processor.size = {
    'shortest_edge': min_vision_tokens * PIXELS_PER_TOKEN,
    'longest_edge': max_vision_tokens * PIXELS_PER_TOKEN,
}
print('Cosmos-Reason2-2B loaded successfully.')
save_status('cosmos_loaded', 'done')

## 5. Dataset

Reuses the same `g1-pick-apple` episode subset from Phase 2/3. `snapshot_download`
caches by content, so if this points at the same `DATA_DIR` as notebook 1 already used,
nothing gets re-downloaded — this just re-derives the file list. Bump `N_EPISODES` here
if you want to score more episodes than Phase 2 originally pulled.

In [ ]:
from huggingface_hub import HfApi, snapshot_download
import json as _json

G1_DATASET_REPO = 'nvidia/PhysicalAI-Robotics-GR00T-Teleop-G1'
TASK_FOLDER = 'g1-pick-apple'
N_EPISODES = 15
TASK_DIR = os.path.join(DATA_DIR, TASK_FOLDER)

if not is_done('g1_meta_downloaded'):
    snapshot_download(
        repo_id=G1_DATASET_REPO, repo_type='dataset',
        allow_patterns=[f'{TASK_FOLDER}/meta/*'], local_dir=DATA_DIR,
    )
    save_status('g1_meta_downloaded', 'done')

with open(os.path.join(TASK_DIR, 'meta', 'info.json')) as f:
    info = _json.load(f)

discarded = set(info.get('discarded_episode_indices', []))
total_episodes = info['total_episodes']

selected_episodes = []
i = 0
while len(selected_episodes) < N_EPISODES and i < total_episodes:
    if i not in discarded:
        selected_episodes.append(i)
    i += 1
print(f'Selected {len(selected_episodes)} episodes:', selected_episodes)

allow_patterns = []
for ep in selected_episodes:
    allow_patterns.append(f'{TASK_FOLDER}/data/chunk-000/episode_{ep:06d}.parquet')
    allow_patterns.append(f'{TASK_FOLDER}/videos/chunk-000/observation.images.ego_view/episode_{ep:06d}.mp4')
snapshot_download(
    repo_id=G1_DATASET_REPO, repo_type='dataset',
    allow_patterns=allow_patterns, local_dir=DATA_DIR,
)
save_status('g1_episodes_downloaded', 'done')

with open(os.path.join(TASK_DIR, 'meta', 'modality.json')) as f:
    modality_spec = _json.load(f)

episodes_meta = {}
with open(os.path.join(TASK_DIR, 'meta', 'episodes.jsonl')) as f:
    for line in f:
        rec = _json.loads(line)
        episodes_meta[rec['episode_index']] = rec

print('Ready:', len(selected_episodes), 'episodes in', TASK_DIR)

## 6. Shared parsing + forward-kinematics helpers

Same functions as notebook 1 Sections 6/6.5 — copied here rather than imported, since a
notebook can't reliably `import` from another `.ipynb`. Keep these two copies in sync by
hand if either changes.

In [ ]:
import pandas as pd
import numpy as np
import cv2
import urllib.request
import pytorch_kinematics as pk

def split_by_modality(vec, spec):
    vec = np.asarray(vec, dtype=np.float32)
    return {name: vec[bounds['start']:bounds['end']] for name, bounds in spec.items()}

G1_URDF_URL = 'https://raw.githubusercontent.com/unitreerobotics/unitree_ros/master/robots/g1_description/g1_29dof_with_hand_rev_1_0.urdf'
G1_URDF_PATH = os.path.join(CKPT_DIR, 'g1_29dof_with_hand_rev_1_0.urdf')

if not os.path.exists(G1_URDF_PATH):
    urllib.request.urlretrieve(G1_URDF_URL, G1_URDF_PATH)
with open(G1_URDF_PATH, 'rb') as f:
    urdf_bytes = f.read()

left_wrist_chain = pk.build_serial_chain_from_urdf(urdf_bytes, 'left_wrist_yaw_link')
right_wrist_chain = pk.build_serial_chain_from_urdf(urdf_bytes, 'right_wrist_yaw_link')

raw_joint_names = info['features']['observation.state']['names']
assert len(raw_joint_names) == 43

def rot_matrix_to_rot6d(R):
    return R[:, :2].T.reshape(-1)

def compute_wrist_eef_9d(chain, raw_state_vec):
    joint_name_to_val = dict(zip(raw_joint_names, raw_state_vec))
    needed_names = chain.get_joint_parameter_names()
    th = [float(joint_name_to_val[name]) for name in needed_names]
    ret = chain.forward_kinematics(th, end_only=False)
    end_link_name = list(ret.keys())[-1]
    m = ret[end_link_name].get_matrix().numpy()[0]
    pos = m[:3, 3]
    rot6d = rot_matrix_to_rot6d(m[:3, :3])
    return np.concatenate([pos, rot6d]).astype(np.float32)

FK_CHAINS = {'left_wrist_eef_9d': left_wrist_chain, 'right_wrist_eef_9d': right_wrist_chain}
FK_KEYS = set(FK_CHAINS.keys())
print('FK helpers ready.')

## 7. Ground-truth sub-goal segmentation

Implements `docs/scoring_rubric.md` Section 2: derive reach/transport/retreat frame
boundaries from a hand-closure signal (mean abs value of the 7 hand-joint state dims per
frame), rather than from any labeled sub-goal annotation the dataset doesn't have. The
threshold is fit from each episode's own signal range, not a fixed hardcoded value. If
no clean grasp+release pair is found, or the resulting phases are wildly unbalanced,
`segmentation_valid=False` and the episode is limited to a reach-only observation point
(handled in Section 9) — matches the rubric's Section 6 manual-review trigger for a bad
segmentation.

In [ ]:
def compute_hand_closure_signal(df, modality_spec, hand_key):
    bounds = modality_spec['state'][hand_key]
    n_frames = len(df)
    signal = np.zeros(n_frames, dtype=np.float64)
    for t in range(n_frames):
        vec = np.asarray(df.iloc[t]['observation.state'], dtype=np.float64)
        signal[t] = np.mean(np.abs(vec[bounds['start']:bounds['end']]))
    return signal

def fit_closure_threshold(signal):
    """Midpoint-of-range threshold, fit per-episode rather than hardcoded — the exact
    fraction (0.4) is a starting point per scoring_rubric.md Section 2 and is exactly the
    kind of numeric cutoff that section says to revisit after the manual spot-check, not
    a value we're claiming is precisely correct yet."""
    return float(signal.min() + 0.4 * (signal.max() - signal.min()))

def find_events(signal, threshold, min_dwell=3):
    above = signal > threshold
    n = len(above)
    grasp_idx = None
    i = 0
    while i < n:
        if above[i] and i + min_dwell <= n and np.all(above[i:i + min_dwell]):
            grasp_idx = i
            break
        i += 1
    if grasp_idx is None:
        return None, None
    release_idx = None
    i = grasp_idx
    while i < n:
        if not above[i] and i + min_dwell <= n and np.all(~above[i:i + min_dwell]):
            release_idx = i
            break
        i += 1
    return grasp_idx, release_idx

def derive_subgoal_segments(df, modality_spec):
    left_signal = compute_hand_closure_signal(df, modality_spec, 'left_hand')
    right_signal = compute_hand_closure_signal(df, modality_spec, 'right_hand')
    left_range = left_signal.max() - left_signal.min()
    right_range = right_signal.max() - right_signal.min()
    if left_range >= right_range:
        combined_signal, active_hand = left_signal, 'left_hand'
    else:
        combined_signal, active_hand = right_signal, 'right_hand'

    threshold = fit_closure_threshold(combined_signal)
    grasp_idx, release_idx = find_events(combined_signal, threshold)
    n_frames = len(df)
    valid = grasp_idx is not None and release_idx is not None
    if valid:
        reach_frac = grasp_idx / n_frames
        transport_frac = (release_idx - grasp_idx) / n_frames
        retreat_frac = (n_frames - release_idx) / n_frames
        if min(reach_frac, transport_frac, retreat_frac) < 0.05:
            valid = False

    return {
        'active_hand': active_hand,
        'threshold': threshold,
        'signal_min': float(combined_signal.min()),
        'signal_max': float(combined_signal.max()),
        'grasp_idx': grasp_idx,
        'release_idx': release_idx,
        'n_frames': n_frames,
        'segmentation_valid': valid,
        'hand_closure_signal': combined_signal.tolist(),
    }

print('Segmentation helpers ready.')

## 8. Build an observation at an arbitrary start frame

Generalizes notebook 1 Section 7 (which always built the observation from frame 0) to
start from any frame index — needed so GR00T can be queried at the transport and retreat
sub-goal starts, not just the episode start. Same key-discovery pattern as before
(reads `gr00t_modality_configs` rather than hardcoding key names), plus a `cv2` seek to
the requested start frame.

In [ ]:
def build_observation(df, video_path, episode_length, start_frame, task_instruction,
                       gr00t_modality_configs, modality_spec, fk_chains, fk_keys):
    video_keys = gr00t_modality_configs['video'].modality_keys
    state_keys = gr00t_modality_configs['state'].modality_keys
    language_keys = gr00t_modality_configs['language'].modality_keys
    T_video = len(gr00t_modality_configs['video'].delta_indices)
    T_state = len(gr00t_modality_configs['state'].delta_indices)
    assert len(video_keys) == 1 and len(language_keys) == 1

    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    frames = []
    for _ in range(T_video):
        ret, f = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    while 0 < len(frames) < T_video:
        frames.append(frames[-1])
    video_arr = np.stack(frames, axis=0)[np.newaxis, ...]
    observation_video = {video_keys[0]: video_arr}

    raw_state_rows = []
    for t in range(start_frame, start_frame + T_state):
        idx = min(t, episode_length - 1)
        raw_state_rows.append(np.asarray(df.iloc[idx]['observation.state'], dtype=np.float64))
    state_split_rows = [split_by_modality(row, modality_spec['state']) for row in raw_state_rows]

    observation_state = {}
    for k in state_keys:
        if k in fk_keys:
            vals = np.stack([compute_wrist_eef_9d(fk_chains[k], row) for row in raw_state_rows], axis=0)
        else:
            vals = np.stack([row[k] for row in state_split_rows], axis=0)
        observation_state[k] = vals[np.newaxis, ...]

    observation_language = {language_keys[0]: [[task_instruction]]}
    return {'video': observation_video, 'state': observation_state, 'language': observation_language}

print('build_observation() ready.')

## 9. Per-episode pipeline

Runs Cosmos-Reason2 once (frame 0, full task) and GR00T at each valid observation point
(reach/transport/retreat), and packages the raw predicted-vs-actual data the scoring
script needs. No scoring judgment happens here — just data collection.

In [ ]:
from PIL import Image
import tempfile

def get_actual_action_window(df, episode_length, start_frame, T_action, modality_spec, fk_chains, fk_keys):
    end_frame = min(start_frame + T_action, episode_length)
    actual = {k: [] for k in list(modality_spec['action'].keys()) + list(fk_keys)}
    for t in range(start_frame, end_frame):
        vec = np.asarray(df.iloc[t]['action'], dtype=np.float64)
        split = split_by_modality(vec, modality_spec['action'])
        for k, v in split.items():
            actual[k].append(v.tolist())
        for k in fk_keys:
            actual[k].append(compute_wrist_eef_9d(fk_chains[k], vec).tolist())
    return actual, end_frame - start_frame

def run_episode(EP, task_folder, task_dir, episodes_meta, modality_spec,
                 gr00t_policy, gr00t_modality_configs, cosmos_model, cosmos_processor,
                 fk_chains, fk_keys):
    parquet_path = os.path.join(task_dir, 'data', 'chunk-000', f'episode_{EP:06d}.parquet')
    video_path = os.path.join(task_dir, 'videos', 'chunk-000', 'observation.images.ego_view', f'episode_{EP:06d}.mp4')
    df = pd.read_parquet(parquet_path)
    episode_length = episodes_meta[EP]['length']
    task_instruction = episodes_meta[EP]['tasks'][0]

    segments = derive_subgoal_segments(df, modality_spec)

    cap = cv2.VideoCapture(video_path)
    ret, frame_bgr = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError(f'Could not read frame 0 of episode {EP} video: {video_path}')
    first_frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    tmp_frame_path = os.path.join(tempfile.gettempdir(), f'cosmos_scoring_ep{EP}.png')
    Image.fromarray(first_frame_rgb).save(tmp_frame_path)

    reasoning_prompt = (
        f'Task: "{task_instruction}". Describe the sub-goals needed to complete this task, '
        f'in order, as a short numbered list.'
    )
    conversation = [
        {'role': 'system', 'content': [{'type': 'text', 'text': 'You are a helpful assistant.'}]},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': tmp_frame_path},
            {'type': 'text', 'text': reasoning_prompt},
        ]},
    ]
    cosmos_inputs = cosmos_processor.apply_chat_template(
        conversation, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt',
    ).to(cosmos_model.device)
    with torch.no_grad():
        cosmos_out_ids = cosmos_model.generate(**cosmos_inputs, max_new_tokens=200)
    cosmos_out_trimmed = cosmos_out_ids[:, cosmos_inputs['input_ids'].shape[1]:]
    cosmos_reasoning_text = cosmos_processor.batch_decode(
        cosmos_out_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False,
    )[0]

    observation_points = [('reach', 0)]
    if segments['segmentation_valid']:
        observation_points.append(('transport', segments['grasp_idx']))
        observation_points.append(('retreat', segments['release_idx']))

    phase_results = {}
    for phase_name, start_frame in observation_points:
        observation = build_observation(
            df, video_path, episode_length, start_frame, task_instruction,
            gr00t_modality_configs, modality_spec, fk_chains, fk_keys,
        )
        gr00t_action, _ = gr00t_policy.get_action(observation)
        T_action = next(iter(gr00t_action.values())).shape[1]
        actual_action, actual_len = get_actual_action_window(
            df, episode_length, start_frame, T_action, modality_spec, fk_chains, fk_keys,
        )
        predicted_action = {k: v[0, :actual_len, :].tolist() for k, v in gr00t_action.items()}
        phase_results[phase_name] = {
            'start_frame': start_frame,
            'T_action': T_action,
            'compared_steps': actual_len,
            'predicted_action': predicted_action,
            'actual_action': actual_action,
        }

    return {
        'task_folder': task_folder,
        'episode_index': EP,
        'task_instruction': task_instruction,
        'episode_length': episode_length,
        'segmentation': segments,
        'cosmos_reasoning_proxy_text': cosmos_reasoning_text,
        'phase_results': phase_results,
        'wrist_eef_9d_note': (
            "left_wrist_eef_9d / right_wrist_eef_9d computed via forward kinematics "
            "(public Unitree G1 URDF); approximation, not verified against NVIDIA's "
            "internal training convention."
        ),
    }

print('run_episode() ready.')

## 10. Main loop — run across selected episodes

Checkpointed per episode: an episode whose JSON already exists in `RAW_LOG_DIR` is
skipped, so a disconnected session resumes cleanly without redoing finished episodes.

In [ ]:
for EP in selected_episodes:
    out_path = os.path.join(RAW_LOG_DIR, f'episode_{EP:06d}.json')
    if os.path.exists(out_path):
        print(f'Episode {EP}: already logged, skipping.')
        continue
    print(f'Episode {EP}: running...')
    result = run_episode(
        EP, TASK_FOLDER, TASK_DIR, episodes_meta, modality_spec,
        gr00t_policy, gr00t_modality_configs, cosmos_model, cosmos_processor,
        FK_CHAINS, FK_KEYS,
    )
    with open(out_path, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'Episode {EP}: done. segmentation_valid={result["segmentation"]["segmentation_valid"]}, '
          f'phases_logged={list(result["phase_results"].keys())}')

save_status('scoring_pass_1', 'done')
print('\nAll episodes processed. Raw logs at:', RAW_LOG_DIR)

## 11. Status summary + next step

Once this finishes, sync the `raw_logs/` folder from Drive to the local repo (or point
`scripts/score_trajectories.py` directly at the Drive path if running locally with Drive
mounted some other way), then run the scoring script — it needs no GPU, only an
Anthropic API key for the reasoning-stage LLM judge.

In [ ]:
print(json.dumps(load_status(), indent=2))
n_logged = len([f for f in os.listdir(RAW_LOG_DIR) if f.endswith('.json')])
print(f'\n{n_logged} episode logs in {RAW_LOG_DIR}')